In [1]:
import pandas as pd
import numpy as np
import pickle
import wandb
from pathlib import Path
from collections import Counter
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

In [3]:
# Device setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch version : 2.10.0+cu128
Device          : cuda
CUDA available  : True
GPU: Tesla T4


In [5]:
CFG = {
    # Vocabulary
    'vocab_size'    : 15000,   # how many unique words to keep
    'max_len'       : 256,     # max tokens per sample

    # Model architecture
    'embed_dim'     : 128,     # word embedding dimensions
    'hidden_dim'    : 256,     # LSTM hidden state size
    'num_layers'    : 2,       # number of LSTM layers
    'dropout'       : 0.3,     # dropout between LSTM layers
    'num_classes'   : 5,       # A, B, C, D, E

    # Training
    'batch_size'    : 32,
    'learning_rate' : 1e-3,
    'epochs'        : 50,
    'patience'      : 8,       # early stopping patience
    'weight_decay'  : 1e-4,    # L2 regularization
    'random_state'  : 42,

    # Paths
    'data_dir'   : '/kaggle/input/competitions/smart-mcq-solver-challenge',
    'output_dir' : '/kaggle/working/outputs',
}

In [6]:
torch.manual_seed(CFG['random_state'])
np.random.seed(CFG['random_state'])

OUTPUT_DIR = Path(CFG['output_dir'])
DATA_DIR   = Path(CFG['data_dir'])

print('Config set!')
for k, v in CFG.items():
    print(f'   {k:<16}: {v}')

Config set!
   vocab_size      : 15000
   max_len         : 256
   embed_dim       : 128
   hidden_dim      : 256
   num_layers      : 2
   dropout         : 0.3
   num_classes     : 5
   batch_size      : 32
   learning_rate   : 0.001
   epochs          : 50
   patience        : 8
   weight_decay    : 0.0001
   random_state    : 42
   data_dir        : /kaggle/input/competitions/smart-mcq-solver-challenge
   output_dir      : /kaggle/working/outputs


In [7]:
raw_train = pd.read_csv(DATA_DIR / 'train.csv')
raw_test  = pd.read_csv(DATA_DIR / 'test.csv')

In [8]:
# Lowercase all the text
text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in text_cols:
    raw_train[col] = raw_train[col].str.lower().str.strip()
    raw_test[col]  = raw_test[col].str.lower().str.strip()

In [9]:
# Stratified 80/20 split
np.random.seed(CFG['random_state'])
train_idx, val_idx = [], []
for ans in 'ABCDE':
    idx = raw_train[raw_train['answer'] == ans].index.tolist()
    np.random.shuffle(idx)
    cut = int(len(idx) * 0.8)
    train_idx += idx[:cut]
    val_idx   += idx[cut:]

train_df = raw_train.loc[train_idx].reset_index(drop=True)
val_df   = raw_train.loc[val_idx].reset_index(drop=True)
test_df  = raw_test.copy()

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Train: 1599 | Val: 401 | Test: 500


In [10]:
# Answer label maps
ANSWER_MAP  = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
REVERSE_MAP = {v: k for k, v in ANSWER_MAP.items()}